In [9]:
import tkinter as tk
from tkinter import filedialog
from docx import Document
import pandas as pd
import re

def find_korean_in_word(file_path):
    doc = Document(file_path)
    results = []
    korean_pattern = re.compile('[\uac00-\ud7a3]+')
    for i, para in enumerate(doc.paragraphs):
        matches = korean_pattern.findall(para.text)
        if matches:
            for match in matches:
                results.append(f"문단 {i+1}에 한글: '{match}' 포함")
    return results

def find_korean_in_excel(file_path):
    results = []
    korean_pattern = re.compile('[\uac00-\ud7a3]+')
    xls = pd.ExcelFile(file_path)
    for sheet_name in xls.sheet_names:
        sheet_df = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')
        for row_idx, row in sheet_df.iterrows():
            for col_idx, cell in enumerate(row):
                if isinstance(cell, str):
                    matches = korean_pattern.findall(cell)
                    if matches:
                        for match in matches:
                            cell_ref = f"{sheet_name} 시트 {row_idx+1}행 {col_idx+1}열"
                            results.append(f"{cell_ref}에 한글: '{match}' 포함")
    return results

def open_word_file():
    file_path = filedialog.askopenfilename(filetypes=[("Word Files", "*.docx")])
    if file_path:
        results = find_korean_in_word(file_path)
        result_text.delete(1.0, tk.END)
        if results:
            result_text.insert(tk.END, f"[Word 파일] {file_path}\n")
            for res in results:
                result_text.insert(tk.END, res + "\n")
        else:
            result_text.insert(tk.END, "한글이 포함된 문단이 없습니다.\n")

def open_excel_file():
    file_path = filedialog.askopenfilename(filetypes=[("Excel Files", "*.xlsx *.xls")])
    if file_path:
        results = find_korean_in_excel(file_path)
        result_text.delete(1.0, tk.END)
        if results:
            result_text.insert(tk.END, f"[Excel 파일] {file_path}\n")
            for res in results:
                result_text.insert(tk.END, res + "\n")
        else:
            result_text.insert(tk.END, "한글이 포함된 셀이 없습니다.\n")

# GUI 설정
root = tk.Tk()
root.title("한글 위치 체크 프로그램")

btn_word = tk.Button(root, text="워드 파일 선택", command=open_word_file)
btn_word.pack(pady=5)

btn_excel = tk.Button(root, text="엑셀 파일 선택", command=open_excel_file)
btn_excel.pack(pady=5)

result_text = tk.Text(root, height=20, width=80)
result_text.pack(pady=10)

root.mainloop()